In [ ]:
# Otto

In [ ]:
# read in shapefile

In [ ]:
import xarray as xr
import rioxarray as rxr  # activates .rio accessor
import fiona
from pathlib import Path

# ---------------- PATHS ----------------
BASE = Path("/uufs/chpc.utah.edu/common/home/skiles-group4")

TOPO_NC = BASE / "otto_from_skiles-group2/otto2/GSLB_iSnobal_data/Outputs/Jordan/topo.nc"
SHAPE   = BASE / "otto_from_skiles-group2/otto2/GSLB_iSnobal_data/Shapefiles/Central_Wasatch.shp"

OUT_NC  = BASE / "otto_from_skiles-group2/otto2/GSLB_iSnobal_data/Outputs/Jordan/topo_central_wasatch_rect_fix.nc"

# ---------------- LOAD ORIGINAL TOPO ----------------
ds_orig = xr.open_dataset(TOPO_NC)

# Ensure spatial dims for rioxarray
ds = ds_orig.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)

# ---- Get CRS WKT from the grid_mapping variable 'projection' ----
proj_var = ds.get("projection")
if proj_var is not None:
    wkt = proj_var.attrs.get("spatial_ref") or proj_var.attrs.get("crs_wkt")
else:
    wkt = None

if not wkt:
    raise RuntimeError("Could not find spatial_ref/crs_wkt in 'projection' variable.")

# Tell rioxarray the CRS using the WKT string (no pyproj EPSG lookup)
ds = ds.rio.write_crs(wkt, inplace=False)

print("Topo CRS (from WKT via rasterio):", ds.rio.crs)

# ---------------- LOAD SHAPE BBOX (NO geopandas / pyproj) ----------------
with fiona.open(SHAPE) as src:
    minx, miny, maxx, maxy = src.bounds
    print("Shapefile bounds:", src.bounds)

# ---------------- CLIP TO BBOX ----------------
ds_clipped = ds.rio.clip_box(minx=minx, miny=miny, maxx=maxx, maxy=maxy)

# ---------------- RESTORE projection ATTRS (utm_zone_number, etc.) ----------------
if "projection" not in ds_clipped:
    raise RuntimeError("No 'projection' variable in clipped topo!")

if "projection" in ds_orig:
    ds_clipped["projection"].attrs = ds_orig["projection"].attrs.copy()
else:
    # Fallback: just set UTM zone manually if original somehow lacked it
    ds_clipped["projection"].attrs["utm_zone_number"] = 12

# ---------------- CLEAN UP grid_mapping BEFORE SAVING ----------------
# xarray's CF encoder chokes if grid_mapping is in both attrs and encoding.
clean = ds_clipped.copy()

for name, da in list(clean.data_vars.items()):
    da = da.copy()
    da.attrs.pop("grid_mapping", None)
    da.encoding.pop("grid_mapping", None)
    clean[name] = da

for name, da in list(clean.coords.items()):
    if isinstance(da, xr.DataArray):
        da = da.copy()
        da.attrs.pop("grid_mapping", None)
        da.encoding.pop("grid_mapping", None)
        clean.coords[name] = da

# ---------------- SAVE OUT ----------------
clean.to_netcdf(OUT_NC)

print("Wrote fixed, clipped topo to:", OUT_NC)
print(clean)